In [1]:
import os

N_THREADS = 8
os.environ["OMP_NUM_THREADS"] = str(N_THREADS)
os.environ["MKL_NUM_THREADS"] = str(N_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_THREADS)


import gc
import time
import statistics as stats
from dataclasses import dataclass
from typing import Callable, List, Tuple, Dict, Any, Sequence, Union, Optional, Iterable
from pathlib import Path
from __future__ import annotations

import cv2
import numpy as np
import torch
from torch.profiler import profile, ProfilerActivity
import onnxruntime as ort

from SSD_from_scratch import mySSD
import CarImageClass
from SSDInt8_ONNX_Pred import SSDInt8ONNXPredictor, PreprocessConfig

# desktop or laptop
machine = 'laptop'

# Setup path to data folder
if machine == 'laptop':
    folder_path = Path(r"C:\self-driving-car\data")
else:
    folder_path = Path(r"C:\Udacity_car_data\data")

train_path = folder_path / "train"
test_path = folder_path / "test"



# os.environ["OMP_NUM_THREADS"] = "8"
# os.environ["MKL_NUM_THREADS"] = "8"

torch.set_num_threads(N_THREADS)
torch.set_num_interop_threads(1)

In [2]:
def _ms(ns: int) -> float:
    return ns / 1e6

def _pct(sorted_vals: List[float], p: float) -> float:
    # p in [0,100]
    if not sorted_vals:
        return float("nan")
    k = (len(sorted_vals) - 1) * (p / 100.0)
    f = int(k)
    c = min(f + 1, len(sorted_vals) - 1)
    if f == c:
        return sorted_vals[f]
    return sorted_vals[f] + (k - f) * (sorted_vals[c] - sorted_vals[f])

@dataclass
class StageStats:
    n: int
    mean_ms: float
    median_ms: float
    p90_ms: float
    p95_ms: float
    p99_ms: float
    min_ms: float
    max_ms: float

def summarize(times_ms: List[float]) -> StageStats:
    s = sorted(times_ms)
    return StageStats(
        n=len(s),
        mean_ms=sum(s) / len(s),
        median_ms=_pct(s, 50),
        p90_ms=_pct(s, 90),
        p95_ms=_pct(s, 95),
        p99_ms=_pct(s, 99),
        min_ms=s[0],
        max_ms=s[-1],
    )

def bench_inference(
    inputs: Iterable[Any],
    preprocess: Callable[[Any], Any],
    model: mySSD,
    warmup_iters: int = 20,
    measure_iters: int = 200,
) -> Dict[str, StageStats]:
    """
    CPU-only stage timing with warmup + per-iteration latency distributions.

    inputs: iterable of already-in-memory inputs (e.g., numpy arrays or decoded images)
    preprocess: input -> model_input
    forward: model_input -> raw_output
    postprocess: raw_output -> final_output
    """
    inputs = list(inputs)
    if not inputs:
        raise ValueError("inputs must be non-empty (and already in memory).")

    # --- Warmup (stabilizes caches, allocators, thread pools) ---
    wi = 0
    while wi < warmup_iters:
        x = inputs[wi % len(inputs)]
        mi = preprocess(x)
        loc_all, conf_all = model(mi) # forward(mi)
        _ = model.predict(mi, score_thresh=0.3, nms_thresh=0.5, max_per_img=50, pre_loc_all=loc_all, pre_conf_all=conf_all) # postprocess(ro)
        wi += 1

    pre_t, fwd_t, post_t, e2e_t = [], [], [], []

    # --- Measure ---
    for i in range(measure_iters):
        x = inputs[i % len(inputs)]

        t0 = time.perf_counter_ns()
        mi = preprocess(x)
        t1 = time.perf_counter_ns()
        loc_all, conf_all = model(mi) # forward(mi)
        t2 = time.perf_counter_ns()
        _ = model.predict(mi, score_thresh=0.3, nms_thresh=0.5, max_per_img=50, pre_loc_all=loc_all, pre_conf_all=conf_all) # postprocess(ro)
        t3 = time.perf_counter_ns()

        pre_t.append(_ms(t1 - t0))
        fwd_t.append(_ms(t2 - t1))
        post_t.append(_ms(t3 - t2))
        e2e_t.append(_ms(t3 - t0))

    out = {
        "preprocess": summarize(pre_t),
        "forward": summarize(fwd_t),
        "postprocess": summarize(post_t),
        "end_to_end": summarize(e2e_t),
    }

    # Sanity check: end_to_end should roughly equal sum of stages
    # If not, you probably have hidden work outside the stage calls (or timing overhead dominates).
    sum_means = out["preprocess"].mean_ms + out["forward"].mean_ms + out["postprocess"].mean_ms
    if abs(out["end_to_end"].mean_ms - sum_means) / max(out["end_to_end"].mean_ms, 1e-9) > 0.05:
        print(f"[warn] end_to_end mean ({out['end_to_end'].mean_ms:.3f} ms) != sum of means ({sum_means:.3f} ms).")
        print("       This usually indicates hidden work, extra copies, or stage boundary leakage.")

    return out

def print_report(report: Dict[str, StageStats]) -> None:
    for name, s in report.items():
        print(
            f"{name:>11}: n={s.n:4d}  mean={s.mean_ms:8.3f}  "
            f"p50={s.median_ms:8.3f}  p95={s.p95_ms:8.3f}  p99={s.p99_ms:8.3f}  "
            f"min={s.min_ms:8.3f}  max={s.max_ms:8.3f}"
        )

In [2]:
# for loading images

def load_images_uint8_chw(
    folder: str,
    *,
    exts=(".jpg", ".jpeg", ".png"),
    limit: int | None = None,
    convert_to_rgb: bool = False,   # keep False if your preprocessing expects BGR
) -> list[np.ndarray]:
    folder = Path(folder)
    paths = sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])
    if limit is not None:
        paths = paths[:limit]

    imgs: list[np.ndarray] = []
    for p in paths:
        im = cv2.imread(str(p), cv2.IMREAD_COLOR)  # uint8 HWC, BGR
        if im is None:
            raise ValueError(f"Failed to read: {p}")
        if convert_to_rgb:
            im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        # enforce contiguous memory (avoids occasional stride surprises)
        im = np.ascontiguousarray(np.transpose(im, (2, 0, 1)))
        imgs.append(im)

    if not imgs:
        raise ValueError(f"No images found in {folder} with extensions {exts}")
    return imgs

In [3]:
# preprocessing function

def preprocess_uint8_chw_rgb(
    x_chw: Union[np.ndarray, torch.Tensor],
    *,
    out_size: Tuple[int, int] = (300, 300),
    mean: Sequence[float] = (0.485, 0.456, 0.406),
    std: Sequence[float]  = (0.229, 0.224, 0.225),
) -> torch.Tensor:
    """
    Input:
        x_chw: uint8 (C,H,W), RGB, values in [0,255]
    Output:
        torch.float32 (1,C,out_H,out_W)

    Equivalent to:
        v2.ToImage()
        v2.ToDtype(torch.float32, scale=True)
        v2.Resize((300,300), antialias=True)
        v2.Normalize(mean=..., std=...)
    """
    # Convert to torch.Tensor
    if isinstance(x_chw, np.ndarray):
        x = torch.from_numpy(x_chw)
    elif isinstance(x_chw, torch.Tensor):
        x = x_chw
    else:
        raise TypeError(f"Expected numpy.ndarray or torch.Tensor, got {type(x_chw)}")

    if x.ndim != 3:
        raise ValueError(f"Expected shape (C,H,W), got {tuple(x.shape)}")
    if x.dtype != torch.uint8:
        raise TypeError(f"Expected dtype uint8, got {x.dtype}")

    # ToDtype(float32, scale=True) for uint8 => /255
    x = x.to(torch.float32) / 255.0

    # Add batch dim: (1,C,H,W)
    x = x.unsqueeze(0)

    # Resize to (300,300) with antialiasing (bilinear like torchvision for tensors)
    try:
        x = torch.nn.functional.interpolate(x, size=out_size, mode="bilinear", align_corners=False, antialias=True)
    except TypeError:
        # Fallback if antialias not supported in your PyTorch version
        x = torch.nn.functional.interpolate(x, size=out_size, mode="bilinear", align_corners=False)

    # Normalize
    mean_t = torch.tensor(mean, dtype=torch.float32, device=x.device).view(1, -1, 1, 1)
    std_t  = torch.tensor(std,  dtype=torch.float32, device=x.device).view(1, -1, 1, 1)
    x = (x - mean_t) / std_t

    return x

In [4]:
# load PyTorch SSD model
ssd_model_noZO_BS = mySSD(class_to_idx_dict={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                          in_channels=3,
                          variances=(0.1, 0.2))
# WEIGHTS_PATH = r"C:\Users\eblac\Documents\GitHub\self-driving-car\app_files\saved_models\noZoomOut_Bootstrap.pth"
WEIGHTS_PATH = r"C:\Users\eblac\OneDrive\Documents\GitHub\self-driving-car\app_files\saved_models\noZoomOut_Bootstrap.pth"
state_dict = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
ssd_model_noZO_BS.load_state_dict(state_dict, strict=False)
ssd_model_noZO_BS.to(device='cpu');


# load images
img_list = np.array(load_images_uint8_chw(folder=test_path, convert_to_rgb=True, limit=1000))

In [10]:
report = bench_inference(
    inputs=img_list,
    preprocess=preprocess_uint8_chw_rgb,
    model=ssd_model_noZO_BS,
    warmup_iters=30,
    measure_iters=500,
)
print_report(report)

 preprocess: n= 500  mean=   2.158  p50=   1.776  p95=   3.361  p99=   7.810  min=   1.420  max=  39.119
    forward: n= 500  mean= 320.090  p50= 308.262  p95= 387.963  p99= 463.896  min= 292.561  max= 537.851
postprocess: n= 500  mean=   8.045  p50=   6.735  p95=  15.744  p99=  19.945  min=   1.707  max=  38.635
 end_to_end: n= 500  mean= 330.293  p50= 319.541  p95= 400.502  p99= 475.797  min= 298.172  max= 547.319


In [11]:
with torch.inference_mode():
    with profile(activities=[ProfilerActivity.CPU], record_shapes=True, profile_memory=True) as prof:
        for _ in range(50):
            mi = preprocess_uint8_chw_rgb(img_list[0])
            loc_all, conf_all = ssd_model_noZO_BS(mi)
            _ = ssd_model_noZO_BS.predict(mi, score_thresh=0.3, nms_thresh=0.5, max_per_img=50, pre_loc_all=loc_all, pre_conf_all=conf_all)

print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=25))

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
         aten::mkldnn_convolution        82.69%       12.582s        82.89%       12.612s      10.510ms       4.72 Gb           0 b          1200  
    aten::max_pool2d_with_indices         5.79%     881.503ms         5.79%     881.503ms       4.408ms       1.52 Gb       1.52 Gb           200  
          aten::native_batch_norm         4.48%     681.982ms         4.60%     700.322ms     700.322us       4.71 Gb      -2.88 Mb          1000  
                 aten::clamp_min_         2.36%     359.099ms         2.36%     359.099ms     326.454us         

In [28]:
onnx_path = r"C:\Users\eblac\OneDrive\Documents\GitHub\self-driving-car\PTQ_testing\ssd.onnx"

so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# Optional but important for reproducible CPU timings:
so.intra_op_num_threads = 1
so.inter_op_num_threads = 1
# (You can raise intra_op_num_threads later; keep it fixed while profiling.)

sess = ort.InferenceSession(
    onnx_path,
    sess_options=so,
    providers=["CPUExecutionProvider"],
)

# Inspect input/output names, shapes, dtypes
for i, inp in enumerate(sess.get_inputs()):
    print(f"input[{i}] name={inp.name}, shape={inp.shape}, type={inp.type}")

for i, out in enumerate(sess.get_outputs()):
    print(f"output[{i}] name={out.name}, shape={out.shape}, type={out.type}")

input[0] name=images, shape=['batch', 3, 300, 300], type=tensor(float)
output[0] name=loc, shape=['batch', 8732, 4], type=tensor(float)
output[1] name=conf, shape=['batch', 8732, 6], type=tensor(float)


In [33]:
# Example: convert torch tensor -> numpy float32
mi_t = preprocess_uint8_chw_rgb(img_list[0])           # torch tensor
mi_np = mi_t.detach().cpu().numpy().astype(np.float32) # numpy

input_name = sess.get_inputs()[0].name

# Warmup + run
for _ in range(10):
    _ = sess.run(None, {input_name: mi_np})

loc_all_onnx, conf_all_onnx = sess.run(['loc', 'conf'], {input_name: mi_np})

ssd_model_noZO_BS.eval()
with torch.inference_mode():
    loc_all_pyt, conf_all_pyt = ssd_model_noZO_BS(mi_t)

In [34]:
loc_all_onnx

array([[[ 0.57077   ,  0.5362505 , -2.006925  , -1.1355766 ],
        [ 0.60852575,  0.6881658 , -1.2832962 , -0.3280177 ],
        [ 0.51597357,  0.32630104, -2.2394266 , -0.5545898 ],
        ...,
        [ 0.00806759,  0.5103041 , -0.82010347, -0.67079324],
        [ 0.00751076,  0.502986  , -0.99169236,  0.08169436],
        [ 0.01629372,  1.0574564 , -0.54932046, -1.6516912 ]]],
      dtype=float32)

In [35]:
loc_all_pyt

tensor([[[ 0.5708,  0.5362, -2.0069, -1.1356],
         [ 0.6085,  0.6882, -1.2833, -0.3280],
         [ 0.5160,  0.3263, -2.2394, -0.5546],
         ...,
         [ 0.0081,  0.5103, -0.8201, -0.6708],
         [ 0.0075,  0.5030, -0.9917,  0.0817],
         [ 0.0163,  1.0575, -0.5493, -1.6517]]])

In [5]:
# --- preprocess ONCE ---
ssd_model_noZO_BS.eval()

mi_t = preprocess_uint8_chw_rgb(img_list[0])          # torch tensor, CPU
# Ensure float32 and contiguous
mi_t = mi_t.to(dtype=torch.float32).contiguous()

# Make a numpy view of the SAME values (one-time conversion)
mi_np = np.ascontiguousarray(mi_t.detach().cpu().numpy())


so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.intra_op_num_threads = N_THREADS
so.inter_op_num_threads = 1

onnx_path = r"C:\Users\eblac\OneDrive\Documents\GitHub\self-driving-car\PTQ_testing\ssd.onnx"

sess = ort.InferenceSession(onnx_path, sess_options=so, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
output_names = [o.name for o in sess.get_outputs()]

feeds = {input_name: mi_np}  # reuse dict to avoid per-iter overhead

In [6]:
def bench(fn, warmup=20, iters=200):
    # warmup
    for _ in range(warmup):
        fn()

    times = []
    for _ in range(iters):
        t0 = time.perf_counter_ns()
        fn()
        t1 = time.perf_counter_ns()
        times.append((t1 - t0) / 1e6)  # ms

    times.sort()
    def pct(p):
        k = (len(times) - 1) * p / 100.0
        f = int(k)
        c = min(f + 1, len(times) - 1)
        return times[f] if f == c else times[f] + (k - f) * (times[c] - times[f])

    return {
        "mean_ms": sum(times) / len(times),
        "p50_ms": pct(50),
        "p95_ms": pct(95),
        "p99_ms": pct(99),
        "fps_p50": 1000.0 / pct(50),
    }

In [7]:
with torch.inference_mode():
    def pyt_forward():
        _ = ssd_model_noZO_BS(mi_t)

def ort_forward():
    _ = sess.run(output_names, feeds)

pyt_stats = bench(pyt_forward, warmup=50, iters=300)
ort_stats = bench(ort_forward, warmup=50, iters=300)

print("PyTorch:", pyt_stats)
print("ONNXRT :", ort_stats)

PyTorch: {'mean_ms': 319.9034943333336, 'p50_ms': 313.5236, 'p95_ms': 355.57193000000007, 'p99_ms': 436.12098099999986, 'fps_p50': 3.1895525568091205}
ONNXRT : {'mean_ms': 230.36802033333342, 'p50_ms': 216.35365000000002, 'p95_ms': 308.175935, 'p99_ms': 350.0047349999999, 'fps_p50': 4.6220620729070205}
